# Tests: `fasterai.sparse.sparsifier` (source `nbs/sparse/sparsifier.ipynb`)

In [ ]:
from fastcore.test import *
import contextlib, io, os, tempfile
import torch
import torch.nn as nn
from torch.nn.utils import parametrize
from fasterai.core.criteria import activation_criteria, large_final
from fasterai.core.parametrize import _is_parametrized, _master
from fasterai.sparse.sparsifier import *

In [ ]:
from fastcore.test import *
import warnings

def _test_model():
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.Conv2d(16, 32, 3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(32, 10)
    )

def _zeros_pct(model):
    z = sum((m.weight==0).sum().item() for m in model.modules() if isinstance(m, nn.Conv2d))
    t = sum(m.weight.numel() for m in model.modules() if isinstance(m, nn.Conv2d))
    return 100 * z / t

# A fraction means what it says: 0.5 zeroes half the weights
conv = nn.Conv2d(3, 16, 3)
sp = Sparsifier(nn.Sequential(conv), 'weight', 'local', large_final, layer_type=nn.Conv2d)
sp.sparsify_layer(conv, 0.5)
test_close((conv.weight == 0).float().mean().item() * 100, 50.0, eps=5.0)

# Buffers created
assert hasattr(conv, '_mask')
assert hasattr(conv, '_init_weights')

# Clean buffers
sp._clean_buffers()
assert not hasattr(conv, '_mask')

# sparsify_model with a fraction
model = _test_model()
Sparsifier(model, 'weight', 'local', large_final).sparsify_model(0.3)
test_close(_zeros_pct(model), 30.0, eps=5.0)

# A percent is read as x/100 for one release, and warns
_pmodel = _test_model()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    Sparsifier(_pmodel, 'weight', 'local', large_final).sparsify_model(50)
test_close(_zeros_pct(_pmodel), 50.0, eps=5.0)
test_eq({x.category for x in w}, {FutureWarning})
assert 'looks like a percent' in str(w[0].message)

# A fraction never warns
_fmodel = _test_model()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    Sparsifier(_fmodel, 'weight', 'local', large_final).sparsify_model(0.5)
test_eq(len(w), 0)

# Dict + context='global' raises ValueError
model2 = _test_model()
sp_g = Sparsifier(model2, 'weight', 'global', large_final)
with ExceptionExpected(ValueError):
    sp_g.sparsify_model({'0': 0.3, '3': 0.6})

# Per-layer dict: each layer reaches its own fraction, and a bad value names its layer
model_d = _test_model()
Sparsifier(model_d, 'weight', 'local', large_final).sparsify_model({'0': 0.3, '3': 0.6})
test_close((model_d[0].weight==0).float().mean().item(), 0.3, eps=0.05)
test_close((model_d[3].weight==0).float().mean().item(), 0.6, eps=0.05)
with ExceptionExpected(ValueError, regex="'3'"):
    Sparsifier(_test_model(), 'weight', 'local', large_final).sparsify_model({'0': 0.3, '3': 150})

# Invalid sparsity
model3 = _test_model()
sp3 = Sparsifier(model3, 'weight', 'local', large_final)
conv3 = nn.Conv2d(3, 16, 3)
with ExceptionExpected(ValueError): sp3.sparsify_layer(conv3, 150)
with ExceptionExpected(ValueError): sp3.sparsify_layer(conv3, -0.1)
with ExceptionExpected(TypeError): sp3.sparsify_layer(conv3, True)
with ExceptionExpected(TypeError): sp3.sparsify_model('0.4')

# print_sparsity runs without error
model4 = _test_model()
sp4 = Sparsifier(model4, 'weight', 'local', large_final)
sp4.sparsify_model(0.5)
sp4.print_sparsity()

# save_model writes a buffer-free model (regression: `copy` used to be missing at import time)
import tempfile, os
model5 = _test_model()
sp5 = Sparsifier(model5, 'weight', 'local', large_final)
sp5.sparsify_model(0.5)
with tempfile.TemporaryDirectory() as _d:
    _path = os.path.join(_d, 'ticket.pth')
    sp5.save_model(_path)
    assert os.path.exists(_path)
    _reloaded = torch.load(_path, weights_only=False)
for m in _reloaded.modules():
    assert not hasattr(m, '_mask') and not hasattr(m, '_init_weights')

# --- Wanda with Sparsifier ---
_wdata = [torch.randn(4, 3, 8, 8)]

# Wanda without data raises ValueError
with ExceptionExpected(ValueError):
    Sparsifier(_test_model(), 'weight', 'local', activation_criteria(torch.abs))

# Wanda with data works
_wmodel2 = _test_model()
sp_w = Sparsifier(_wmodel2, 'weight', 'local', activation_criteria(torch.abs), data=_wdata)
sp_w.sparsify_model(0.5)
test_close(_zeros_pct(_wmodel2), 50.0, eps=5.0)

In [ ]:
# --- a parametrized weight (what `FakeQuantizeCallback` installs): every site reads the MASTER ---
class _Double(nn.Module):
    "A parametrization with a visible effect: the weight this module computes is twice its master"
    def forward(self, w): return w * 2

def _parametrized_model():
    "A model whose first convolution and whose Linear compute their weight, and whose BatchNorms do not"
    m = _test_model()
    parametrize.register_parametrization(m[0], 'weight', _Double())
    parametrize.register_parametrization(m[8], 'weight', _Double())
    return m

_x = torch.randn(2, 3, 8, 8)
_m = _parametrized_model()
_sp = Sparsifier(_m, 'weight', 'local', large_final)   # layer_type=nn.Conv2d: the Linear is not sparsified

# both filter branches walk the model's own modules: the `parametrizations` ModuleDict answers
# `hasattr(m, 'weight')` with a ParametrizationList, and must never be taken for a layer
test_eq(list(_sp._iter_layers('has_weight')), [_m[0], _m[1], _m[3], _m[4], _m[8]])
test_eq(list(_sp._iter_layers()), [_m[0], _m[3]])

# _save_weights snapshots the master of a parametrized module, and the weight of a plain one
test_eq(torch.equal(_m[0]._init_weights, _master(_m[0]).detach()), True)
test_ne(_m[0].weight.detach().tolist(), _m[0]._init_weights.tolist())
test_eq(torch.equal(_m[1]._init_weights, _m[1].weight.detach()), True)
# ...and the containers a parametrization inserts hold no weight of their own to snapshot
test_eq([n for n, _ in _m.named_buffers() if 'parametrizations' in n], [])

_sp.sparsify_model(0.5)
test_close((_master(_m[0]) == 0).float().mean().item(), 0.5, eps=0.1)
test_close((_m[0].weight == 0).float().mean().item(), 0.5, eps=0.1)   # the computed weight shows the mask

# print_sparsity counts the master: the mask, not the zeros a rounding would add of its own
_out = io.StringIO()
with contextlib.redirect_stdout(_out): _sp.print_sparsity()
_overall = [l for l in _out.getvalue().splitlines() if l.startswith('Overall')][0]
test_close(float(_overall.rstrip('%').split()[-1]), 100 * (_master(_m[0]) == 0).float().mean().item(),
           eps=5.0)

# the rewind writes the master: a write to `m.weight` would be discarded by the next forward
with torch.no_grad(): _master(_m[8]).mul_(0.5)
_sp._reset_weights()
test_eq(torch.equal(_master(_m[8]).detach(), _m[8]._init_weights), True)
test_eq(torch.equal(_master(_m[0]).detach(), _m[0]._init_weights * _m[0]._mask), True)

# save_model writes the floating-point master, and leaves the live model working: torch deletes the
# `weight` property from a class the copy SHARES, so the strip has to give the plain class back
_ref = _m(_x).detach().clone()
with tempfile.TemporaryDirectory() as _d:
    _path = os.path.join(_d, 'ticket.pth')
    _sp.save_model(_path, _m)
    _ticket = torch.load(_path, weights_only=False)
test_eq(_is_parametrized(_ticket[0]), False)
test_eq(torch.equal(_ticket[8].weight.detach(), _m[8]._init_weights), True)
for _mod in _ticket.modules():
    assert not hasattr(_mod, '_mask') and not hasattr(_mod, '_init_weights')
test_eq(_ticket(_x).shape, (2, 10))
test_eq(_is_parametrized(_m[0]), True)
test_eq(torch.equal(_m(_x), _ref), True)

# the BatchNorm heuristic still pairs a convolution with the module registered after it, which the
# parametrization's own containers now sit between
_m = _parametrized_model()
Sparsifier(_m, 'filter', 'local', large_final).sparsify_model(0.5)
assert (_m[1].weight == 0).sum().item() > 0, 'the BatchNorm after the parametrized conv was left alone'
assert (_m[4].weight == 0).sum().item() > 0, 'the BatchNorm after the plain conv was left alone'

In [ ]:
#| slow
from torchvision.models import resnet18
model_lg = resnet18(weights=None)
sp_lg = Sparsifier(model_lg, 'weight', 'local', large_final)
sp_lg.sparsify_model(0.6)
total_zeros = sum((m.weight==0).sum().item() for m in model_lg.modules() if isinstance(m, nn.Conv2d))
total_params = sum(m.weight.numel() for m in model_lg.modules() if isinstance(m, nn.Conv2d))
test_close(100*total_zeros/total_params, 60.0, eps=10.0)